# 🧠 EfficientNet-B3 — Complete Training Pipeline

**Task:** 5-class image classification using transfer learning on a preprocessed dataset.

---

### 📋 Pipeline Sections

| # | Section |
|---|---|
| 1 | Environment Setup & GPU Verification |
| 2 | Imports |
| 3 | Configuration |
| 4 | Dataset Loading |
| 5 | Dataset Validation |
| 6 | Dataset Visualization |
| 7 | Model Creation |
| 8 | Model Compilation |
| 9 | Initial Training (Head Only) |
| 10 | Fine-Tuning (Partial Unfreeze) |
| 11 | Training Curves |
| 12 | Test Evaluation |
| 13 | Confusion Matrix |
| 14 | Error Analysis |
| 15 | Model Saving & Artifacts |
| 16 | Inference Test |
| 17 | Final Results Summary |

## 1 — Environment Setup & GPU Verification

In [ ]:
import sys
print(f"Python version : {sys.version}")

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

# ── GPU check ────────────────────────────────────────────────────────────────
# NOTE: we do NOT import subprocess here — doing so inside a conditional block
#       can trigger Colab's argparse/-f kernel argument error.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU Available: {len(gpus)} device(s) found.")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
else:
    print("\n⚠️  No GPU detected.")
    print("   → Go to: Runtime > Change runtime type > T4 GPU, then restart.")

In [ ]:
# GPU details via shell magic — avoids any Python argparse conflicts
!nvidia-smi

> **Note:** If the cell above shows `nvidia-smi: command not found`, your runtime has no GPU.  
> Go to **Runtime → Change runtime type → T4 GPU** and restart.

In [ ]:
# ── Mount Google Drive ───────────────────────────────────────────────────────
# If you see a SystemExit / argparse / '-f' error here:
#   Step 1 → Runtime > Restart runtime
#   Step 2 → Run THIS cell first (alone), then continue with the rest
#
# The error is a Colab kernel state issue, not a code bug.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive mounted successfully.")
except SystemExit:
    print("⚠️  SystemExit caught — Colab kernel state issue.")
    print("   Fix: Runtime → Restart runtime → re-run this cell.")
except Exception as e:
    print(f"Drive mount error: {e}")
    print("   Fix: Runtime → Restart runtime → re-run this cell.")

In [ ]:
!pip install -q scikit-learn seaborn
print("✅ Packages ready.")

## 2 — Imports

In [ ]:
import os, json, glob, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
)
from tensorflow.keras.models import Sequential

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    precision_recall_fscore_support
)
from PIL import Image
from collections import Counter

warnings.filterwarnings('ignore')
print("✅ All imports successful.")

## 3 — Configuration

In [ ]:
# ── Random Seeds (Reproducibility) ──────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Dataset Paths ────────────────────────────────────────────────────────────
# ‼️  Update DATASET_ROOT to match where you uploaded split_dataset/ in Drive.
# Expected layout:
#   /content/drive/MyDrive/split_dataset/train/{0,1,2,3,4}/
#   /content/drive/MyDrive/split_dataset/val/{0,1,2,3,4}/
#   /content/drive/MyDrive/split_dataset/test/{0,1,2,3,4}/
DATASET_ROOT = "/content/drive/MyDrive/split_dataset"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

# ── Model / Training Config ──────────────────────────────────────────────────
IMAGE_SIZE  = (300, 300)   # EfficientNet-B3 native resolution
BATCH_SIZE  = 16           # 16 on free Colab T4 (16 GB); increase to 32 on A100
NUM_CLASSES = 5

# Phase 1 — train classification head only (backbone frozen)
HEAD_EPOCHS = 20
HEAD_LR     = 1e-3

# Phase 2 — fine-tune upper layers of EfficientNet backbone
FINETUNE_EPOCHS        = 30     # additional epochs
FINETUNE_LR            = 1e-5  # 100x smaller than HEAD_LR
FINETUNE_UNFREEZE_FROM = 200   # unfreeze backbone layers from this index onward
                                # EfficientNetB3 has ~385 layers total

# ── Output Directory ─────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/efficientnet_b3_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_MODEL_PATH   = os.path.join(OUTPUT_DIR, "best_model.keras")
HISTORY_CSV_PATH  = os.path.join(OUTPUT_DIR, "training_history.csv")
METRICS_JSON_PATH = os.path.join(OUTPUT_DIR, "test_metrics.json")
CM_PATH           = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
CM_JSON_PATH      = os.path.join(OUTPUT_DIR, "confusion_matrix.json")
CLASS_NAMES_PATH  = os.path.join(OUTPUT_DIR, "class_names.json")

AUTOTUNE = tf.data.AUTOTUNE

print("✅ Configuration set.")
print(f"   Image size  : {IMAGE_SIZE}")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   Num classes : {NUM_CLASSES}")
print(f"   Output dir  : {OUTPUT_DIR}")

## 4 — Dataset Loading

The dataset is **already split** into `train/`, `val/`, and `test/` — no re-preprocessing needed.

### Pipeline optimizations applied:
- **Prefetching** overlaps CPU preprocessing with GPU training
- **Caching** on val/test avoids repeated disk I/O across epochs
- The loader handles resizing to 300×300; the images are untouched otherwise
- EfficientNet channel normalization lives **inside the model** (applied only during forward pass)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",   # one-hot vectors → matches categorical_crossentropy
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

CLASS_NAMES = train_ds.class_names
print(f"\nClass names detected: {CLASS_NAMES}")

with open(CLASS_NAMES_PATH, 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print(f"Class names saved → {CLASS_NAMES_PATH}")

# Train: prefetch only (caching ~30k images risks OOM on free Colab T4)
train_ds = train_ds.prefetch(AUTOTUNE)

# Val & Test: cache in RAM → zero disk I/O from epoch 2 onward
val_ds  = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

print("\n✅ Datasets loaded and pipelines configured.")

## 5 — Dataset Validation

Before training we verify shapes, label distributions, pixel value ranges, and batch correctness.

In [ ]:
def count_samples_in_split(split_dir, class_names):
    """Count images per class directly from the filesystem."""
    counts = {}
    for cls in class_names:
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            files = [f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
            counts[cls] = len(files)
        else:
            counts[cls] = 0
    return counts

print("=" * 60)
print(" DATASET SAMPLE COUNTS")
print("=" * 60)

split_dirs   = {"train": TRAIN_DIR, "val": VAL_DIR, "test": TEST_DIR}
split_totals = {}

for split_name, split_dir in split_dirs.items():
    counts = count_samples_in_split(split_dir, CLASS_NAMES)
    total  = sum(counts.values())
    split_totals[split_name] = total
    print(f"\n  [{split_name.upper()}]  Total: {total}")
    for cls, cnt in counts.items():
        bar = chr(9608) * (cnt // 200)
        print(f"    Class {cls}: {cnt:5d}  {bar}")

grand_total = sum(split_totals.values())
print(f"\n  GRAND TOTAL: {grand_total}")

print("\n" + "=" * 60)
print(" BATCH SHAPE & VALUE RANGE")
print("=" * 60)

for split_name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    images, labels = next(iter(ds))
    print(f"\n  [{split_name.upper()}]")
    print(f"    Image batch shape : {images.shape}")
    print(f"    Label batch shape : {labels.shape}")
    print(f"    Pixel min  : {tf.reduce_min(images).numpy():.2f}")
    print(f"    Pixel max  : {tf.reduce_max(images).numpy():.2f}")
    print(f"    Pixel mean : {tf.reduce_mean(images).numpy():.2f}")
    print(f"    Label sample (one-hot): {labels[0].numpy()}")
    pred_cls = tf.argmax(labels[0]).numpy()
    print(f"    Inferred class: {pred_cls} → '{CLASS_NAMES[pred_cls]}'")

print("\n✅ Dataset validation passed.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Class Distribution per Split", fontsize=14, fontweight='bold')

split_colors = ["#4C72B0", "#55A868", "#C44E52"]

for ax, (split_name, split_dir), color in zip(axes, split_dirs.items(), split_colors):
    counts = count_samples_in_split(split_dir, CLASS_NAMES)
    bars = ax.bar(counts.keys(), counts.values(), color=color, edgecolor='white', linewidth=0.5)
    ax.set_title(f"{split_name.capitalize()} Split", fontsize=12)
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Samples")
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                str(val), ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, max(counts.values()) * 1.15)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Distribution plot saved.")

## 6 — Dataset Visualization

Sample grid confirming correct image loading and class labelling.

In [ ]:
COLS = 5   # one column per class
ROWS = 3   # sample rows

fig, axes = plt.subplots(ROWS, COLS, figsize=(3 * COLS, 3 * ROWS))
fig.suptitle("Training Samples (EfficientNet preprocessing is applied inside the model)",
             fontsize=11, fontweight='bold')

per_class = {i: [] for i in range(NUM_CLASSES)}
for imgs, lbls in train_ds:
    for img, lbl in zip(imgs.numpy(), lbls.numpy()):
        cls_idx = int(np.argmax(lbl))
        if len(per_class[cls_idx]) < ROWS:
            per_class[cls_idx].append(img.astype('uint8'))
    if all(len(v) >= ROWS for v in per_class.values()):
        break

for row in range(ROWS):
    for col in range(COLS):
        ax = axes[row, col]
        if per_class[col] and row < len(per_class[col]):
            ax.imshow(per_class[col][row])
        ax.axis('off')
        if row == 0:
            ax.set_title(f"Class {CLASS_NAMES[col]}", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_images.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample image grid saved.")

## 7 — Model Creation

### Architecture: EfficientNet-B3 with Transfer Learning

```
Input (300×300×3)
    ↓
preprocess_input()          ← EfficientNet per-channel normalisation
    ↓
Data Augmentation           ← RandomFlip | RandomRotation | RandomZoom | RandomContrast
    ↓                          (active only during training=True)
EfficientNet-B3 backbone    ← ImageNet weights, frozen in Phase 1
    ↓
GlobalAveragePooling2D      → (None, 1536)
    ↓
Dropout(0.4)
    ↓
Dense(256, ReLU) + BatchNormalization
    ↓
Dropout(0.3)
    ↓
Dense(5, Softmax)           → class probabilities
```

**Key design decisions:**
- Augmentation inside the model → automatically skipped during val/test/inference
- BatchNorm in the head → stabilises training when feature magnitudes vary across image sources
- `training=False` on backbone → BN layers use learned statistics even during Phase 1 training

In [ ]:
def build_model(image_size=IMAGE_SIZE, num_classes=NUM_CLASSES, dropout_1=0.4, dropout_2=0.3):
    """Build EfficientNet-B3 transfer-learning model."""

    # Pretrained backbone — no top classification layer
    base_model = EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=(*image_size, 3)
    )
    base_model.trainable = False   # frozen in Phase 1

    # Data augmentation sub-model
    aug = Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ], name="data_augmentation")

    # Functional graph
    inputs = tf.keras.Input(shape=(*image_size, 3), name="input_images")
    x = preprocess_input(inputs)           # EfficientNet normalisation
    x = aug(x)                             # augmentation (training only)
    x = base_model(x, training=False)      # backbone BN always in inference mode
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_1, name="dropout_1")(x)
    x = layers.Dense(256, activation="relu", name="dense_head")(x)
    x = layers.BatchNormalization(name="bn_head")(x)
    x = layers.Dropout(dropout_2, name="dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = Model(inputs, outputs, name="EfficientNetB3_TL")
    return model, base_model


model, base_model = build_model()
model.summary(line_length=90)
print(f"\nTotal layers in base model : {len(base_model.layers)}")

## 8 — Model Compilation

### Class Imbalance Note
Training split is **perfectly balanced** (5,968 samples/class) → no class weighting required.
Val/test have minor imbalance → we report macro-averaged metrics.

In [ ]:
def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

compile_model(model, HEAD_LR)
print(f"✅ Model compiled — Phase 1 LR: {HEAD_LR}")

## 9 — Phase 1: Initial Training (Classification Head Only)

**Goal:** Learn a good classifier on top of frozen ImageNet features.

**Callbacks:**
- `ModelCheckpoint` → saves best model by `val_accuracy`
- `EarlyStopping` → stops if `val_loss` doesn't improve for 6 epochs; restores best weights
- `ReduceLROnPlateau` → halves LR after 3 epochs of plateau (floor: 1e-6)
- `CSVLogger` → appends epoch metrics to CSV for reproducible analysis

In [ ]:
checkpoint_cb = ModelCheckpoint(
    filepath=BEST_MODEL_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop_cb = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

csv_logger_cb = CSVLogger(HISTORY_CSV_PATH, append=True)

CALLBACKS = [checkpoint_cb, early_stop_cb, reduce_lr_cb, csv_logger_cb]
print("✅ Callbacks configured.")

In [ ]:
print("=" * 60)
print(" PHASE 1 — Training Classification Head")
print("=" * 60)
print(f" Frozen backbone | LR={HEAD_LR} | Max epochs={HEAD_EPOCHS}")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=CALLBACKS
)

print("\n✅ Phase 1 training complete.")

## 10 — Phase 2: Fine-Tuning

**Why fine-tune?**
Phase 1 taught the head which ImageNet features are useful.
Now we adapt the upper backbone layers to capture patterns specific to our images.

**Strategy:**
- Unfreeze layers from index `FINETUNE_UNFREEZE_FROM` onward (~30% of backbone)
- LR is **100× smaller** than Phase 1 to preserve pretrained weights
- All **BatchNorm layers remain frozen** to avoid corrupting learned statistics

In [ ]:
base_model.trainable = True

# Freeze layers before the unfreeze boundary
for layer in base_model.layers[:FINETUNE_UNFREEZE_FROM]:
    layer.trainable = False

# Keep all BatchNorm layers frozen throughout fine-tuning
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfrozen backbone layers : {trainable_count} / {len(base_model.layers)}")

compile_model(model, FINETUNE_LR)
print(f"✅ Model recompiled — Phase 2 LR: {FINETUNE_LR}")

trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Total trainable parameters now: {trainable_params:,}")

In [ ]:
initial_epoch = len(history_phase1.epoch)
total_epochs  = initial_epoch + FINETUNE_EPOCHS

print("=" * 60)
print(" PHASE 2 — Fine-Tuning")
print("=" * 60)
print(f" Partially unfrozen | LR={FINETUNE_LR} | Epochs {initial_epoch} -> {total_epochs}")
print("=" * 60)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    callbacks=CALLBACKS
)

print("\n✅ Phase 2 fine-tuning complete.")

## 11 — Training Curves

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

history_all  = merge_histories(history_phase1, history_phase2)
phase1_end   = len(history_phase1.epoch)
epochs_range = range(1, len(history_all['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History — EfficientNet-B3", fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(epochs_range, history_all['accuracy'],     label='Train Accuracy', color='#4C72B0')
ax.plot(epochs_range, history_all['val_accuracy'], label='Val Accuracy',   color='#C44E52', linestyle='--')
ax.axvline(phase1_end, color='gray', linestyle=':', linewidth=1.5, label=f'Fine-tune start (ep {phase1_end})')
ax.set_title('Accuracy'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(epochs_range, history_all['loss'],     label='Train Loss', color='#4C72B0')
ax.plot(epochs_range, history_all['val_loss'], label='Val Loss',   color='#C44E52', linestyle='--')
ax.axvline(phase1_end, color='gray', linestyle=':', linewidth=1.5, label=f'Fine-tune start (ep {phase1_end})')
ax.set_title('Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(history_all['val_accuracy'])
best_ep      = history_all['val_accuracy'].index(best_val_acc) + 1
print(f"\nBest val accuracy : {best_val_acc:.4f}  at epoch {best_ep}")

## 12 — Test Evaluation

We load the **best saved checkpoint** and evaluate it on the held-out test set.
This reports the performance of the model with the highest validation accuracy, not the final epoch.

In [ ]:
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)
print(f"✅ Best model loaded from: {BEST_MODEL_PATH}")

In [ ]:
print("=" * 60)
print(" TEST SET EVALUATION (Keras built-in metrics)")
print("=" * 60)

test_loss, test_acc, test_prec, test_rec = best_model.evaluate(test_ds, verbose=1)

print(f"\n  Test Loss      : {test_loss:.4f}")
print(f"  Test Accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Test Precision : {test_prec:.4f}")
print(f"  Test Recall    : {test_rec:.4f}")
test_f1 = 2 * test_prec * test_rec / (test_prec + test_rec + 1e-7)
print(f"  Test F1-Score  : {test_f1:.4f}")

In [ ]:
# Per-class metrics via sklearn (full label arrays needed)
y_true_all, y_pred_all = [], []

for images, labels in test_ds:
    preds = best_model(images, training=False)
    y_true_all.extend(tf.argmax(labels, axis=1).numpy())
    y_pred_all.extend(tf.argmax(preds,  axis=1).numpy())

y_true_arr = np.array(y_true_all)
y_pred_arr = np.array(y_pred_all)

print("\n" + "=" * 60)
print(" CLASSIFICATION REPORT (per class)")
print("=" * 60)
report = classification_report(
    y_true_arr, y_pred_arr,
    target_names=[f"Class {c}" for c in CLASS_NAMES],
    digits=4
)
print(report)

macro_f1    = f1_score(y_true_arr, y_pred_arr, average='macro')
weighted_f1 = f1_score(y_true_arr, y_pred_arr, average='weighted')
macro_prec  = precision_score(y_true_arr, y_pred_arr, average='macro')
macro_rec   = recall_score(y_true_arr, y_pred_arr, average='macro')
print(f"  Macro F1    : {macro_f1:.4f}")
print(f"  Weighted F1 : {weighted_f1:.4f}")

metrics = {
    "test_loss"         : float(test_loss),
    "test_accuracy"     : float(test_acc),
    "macro_precision"   : float(macro_prec),
    "macro_recall"      : float(macro_rec),
    "macro_f1"          : float(macro_f1),
    "weighted_f1"       : float(weighted_f1),
    "best_val_accuracy" : float(best_val_acc)
}
with open(METRICS_JSON_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n✅ Metrics saved -> {METRICS_JSON_PATH}")

## 13 — Confusion Matrix

In [ ]:
cm      = confusion_matrix(y_true_arr, y_pred_arr)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

labels_cm = [f"Class {c}" for c in CLASS_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix — Test Set", fontsize=14, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_cm, yticklabels=labels_cm, ax=axes[0])
axes[0].set_title("Counts", fontsize=12)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=labels_cm, yticklabels=labels_cm, ax=axes[1])
axes[1].set_title("Row-Normalised (Recall per class)", fontsize=12)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig(CM_PATH, dpi=150, bbox_inches='tight')
plt.show()

with open(CM_JSON_PATH, 'w') as f:
    json.dump(cm.tolist(), f, indent=2)

print("\nTop confusion pairs (excluding diagonal):")
cm_nd = cm.copy(); np.fill_diagonal(cm_nd, 0)
flat_idx = np.argsort(cm_nd.flatten())[::-1][:10]
for idx in flat_idx:
    r, c = divmod(idx, NUM_CLASSES)
    if cm_nd[r, c] > 0:
        print(f"  Actual Class {CLASS_NAMES[r]} -> Predicted Class {CLASS_NAMES[c]}: {cm_nd[r,c]} samples")

print(f"\n✅ Confusion matrix saved -> {CM_PATH}")

## 14 — Error Analysis

Per-class performance breakdown and visual inspection of misclassified samples.

In [ ]:
prec_per, rec_per, f1_per, _ = precision_recall_fscore_support(
    y_true_arr, y_pred_arr, labels=list(range(NUM_CLASSES))
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Per-Class Performance on Test Set", fontsize=13, fontweight='bold')

x = np.arange(NUM_CLASSES)
xlabels = [f"Class {c}" for c in CLASS_NAMES]

for ax, values, title, color in zip(
    axes,
    [prec_per, rec_per, f1_per],
    ["Precision", "Recall", "F1-Score"],
    ["#4C72B0", "#55A868", "#C44E52"]
):
    bars = ax.bar(x, values, color=color, edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(xlabels, rotation=20)
    ax.set_ylim(0, 1.05); ax.set_title(title, fontsize=12); ax.set_ylabel(title)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.axhline(np.mean(values), color='gray', linestyle='--', linewidth=1,
               label=f'Mean={np.mean(values):.3f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "per_class_metrics.png"), dpi=150, bbox_inches='tight')
plt.show()

print("\nWeakest classes by F1-Score:")
for cls_idx, f1_val in sorted(enumerate(f1_per), key=lambda x: x[1]):
    print(f"  Class {CLASS_NAMES[cls_idx]}: F1={f1_val:.4f} | Prec={prec_per[cls_idx]:.4f} | Rec={rec_per[cls_idx]:.4f}")

In [ ]:
MAX_ERRORS_SHOWN = 20
errors = []   # (image_np, true_cls, pred_cls, confidence)

for images, labels in test_ds:
    probs       = best_model(images, training=False).numpy()
    true_cls    = tf.argmax(labels, axis=1).numpy()
    pred_cls    = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)
    for img, tc, pc, conf in zip(images.numpy(), true_cls, pred_cls, confidences):
        if tc != pc:
            errors.append((img.astype('uint8'), tc, pc, float(conf)))
        if len(errors) >= MAX_ERRORS_SHOWN:
            break
    if len(errors) >= MAX_ERRORS_SHOWN:
        break

print(f"Displaying {len(errors)} misclassified samples.")

COLS = 5
ROWS = (len(errors) + COLS - 1) // COLS
fig, axes = plt.subplots(ROWS, COLS, figsize=(3.5 * COLS, 3.5 * ROWS))
fig.suptitle("Misclassified Test Samples — Actual vs Predicted", fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.axis('off')
    if i < len(errors):
        img, tc, pc, conf = errors[i]
        ax.imshow(img)
        ax.set_title(f"True: {CLASS_NAMES[tc]}\nPred: {CLASS_NAMES[pc]}  ({conf:.1%})",
                     fontsize=9, color='red')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "error_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Error analysis saved.")

## 15 — Model Saving & Artifacts

In [ ]:
# Save full history as JSON
history_json_path = os.path.join(OUTPUT_DIR, "history_all.json")
serialisable_history = {k: [float(v) for v in vals] for k, vals in history_all.items()}
with open(history_json_path, 'w') as f:
    json.dump(serialisable_history, f, indent=2)
print(f"History JSON saved -> {history_json_path}")

# Copy all artifacts to Google Drive for persistence
import shutil
DRIVE_OUTPUT = "/content/drive/MyDrive/efficientnet_b3_output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for fname in os.listdir(OUTPUT_DIR):
    shutil.copy2(os.path.join(OUTPUT_DIR, fname), os.path.join(DRIVE_OUTPUT, fname))

print(f"\n✅ All artifacts copied to Google Drive -> {DRIVE_OUTPUT}")
print("\nSaved files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}  ({size/1024:.1f} KB)")

## 16 — Inference Test

A self-contained inference function ready for deployment.
EfficientNet preprocessing is **inside the model** — only a resize is needed externally.

In [ ]:
def load_inference_model(model_path, class_names_path):
    """Load the saved model and class names for inference."""
    model = tf.keras.models.load_model(model_path)
    with open(class_names_path, 'r') as f:
        class_names = json.load(f)
    return model, class_names


def predict_single_image(image_path, model, class_names, image_size=(300, 300)):
    """
    Predict the class of a single image file.

    Parameters
    ----------
    image_path  : str  — absolute path to a JPEG or PNG image
    model       : loaded Keras model
    class_names : list of str — class label strings

    Returns
    -------
    dict  with keys: predicted_class, confidence, all_scores
    """
    img = tf.keras.utils.load_img(image_path, target_size=image_size)
    arr = tf.keras.utils.img_to_array(img)   # float32 [0, 255]
    arr = tf.expand_dims(arr, 0)             # add batch dimension

    probs    = model(arr, training=False).numpy()[0]
    pred_idx = int(np.argmax(probs))

    return {
        "predicted_class": class_names[pred_idx],
        "confidence"     : float(probs[pred_idx]),
        "all_scores"     : {class_names[i]: float(p) for i, p in enumerate(probs)}
    }


# Quick test on a random test-set image
inference_model, infer_class_names = load_inference_model(BEST_MODEL_PATH, CLASS_NAMES_PATH)

test_image_path = None
for cls_dir in os.listdir(TEST_DIR):
    cls_full = os.path.join(TEST_DIR, cls_dir)
    if os.path.isdir(cls_full):
        imgs = [f for f in os.listdir(cls_full) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if imgs:
            test_image_path = os.path.join(cls_full, random.choice(imgs))
            true_label = cls_dir
            break

if test_image_path:
    result = predict_single_image(test_image_path, inference_model, infer_class_names)

    print("=" * 50)
    print(" INFERENCE TEST")
    print("=" * 50)
    print(f"  Image      : {os.path.basename(test_image_path)}")
    print(f"  True label : Class {true_label}")
    print(f"  Predicted  : Class {result['predicted_class']}")
    print(f"  Confidence : {result['confidence']:.4f}  ({result['confidence']*100:.2f}%)")
    print("\n  All class probabilities:")
    for cls, score in result['all_scores'].items():
        bar = chr(9608) * int(score * 30)
        print(f"    Class {cls}: {score:.4f}  {bar}")

    img_display = tf.keras.utils.load_img(test_image_path, target_size=(300, 300))
    plt.figure(figsize=(4, 4))
    plt.imshow(img_display)
    color = 'green' if result['predicted_class'] == true_label else 'red'
    plt.title(
        f"True: Class {true_label} | Pred: Class {result['predicted_class']} ({result['confidence']:.1%})",
        color=color, fontsize=10
    )
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No test image found for demo.")

## 17 — Final Results Summary

In [ ]:
with open(METRICS_JSON_PATH, 'r') as f:
    saved_metrics = json.load(f)
with open(CM_JSON_PATH, 'r') as f:
    cm_loaded = np.array(json.load(f))

total_train = count_samples_in_split(TRAIN_DIR, CLASS_NAMES)
total_val   = count_samples_in_split(VAL_DIR,   CLASS_NAMES)
total_test  = count_samples_in_split(TEST_DIR,  CLASS_NAMES)
grand_total = sum(total_train.values()) + sum(total_val.values()) + sum(total_test.values())

total_params = (sum(tf.size(w).numpy() for w in best_model.trainable_weights) +
                sum(tf.size(w).numpy() for w in best_model.non_trainable_weights))

cm_nd = cm_loaded.copy(); np.fill_diagonal(cm_nd, 0)
worst_pair = np.unravel_index(cm_nd.argmax(), cm_nd.shape)

SEP = "=" * 60
print(f"\n{SEP}")
print("  FINAL RESULTS SUMMARY")
print(SEP)
print("  Dataset")
print(f"  |- Grand total      : {grand_total:,}")
print(f"  |- Train samples    : {sum(total_train.values()):,}  (balanced — {list(total_train.values())[0]} per class)")
print(f"  |- Val samples      : {sum(total_val.values()):,}")
print(f"  |- Test samples     : {sum(total_test.values()):,}")
print(f"  +- Num classes      : {NUM_CLASSES}")
print()
print("  Model")
print(f"  |- Architecture     : EfficientNet-B3 + Custom Head (GAP + Dense(256) + Softmax)")
print(f"  |- Total params     : {total_params:,}")
print(f"  |- Input size       : {IMAGE_SIZE[0]} x {IMAGE_SIZE[1]} x 3")
print(f"  |- Phase 1 epochs   : {len(history_phase1.epoch)}")
print(f"  |- Phase 2 epochs   : {len(history_phase2.epoch)}")
print(f"  +- Total epochs run : {len(history_phase1.epoch) + len(history_phase2.epoch)}")
print()
print("  Validation")
print(f"  +- Best val accuracy: {saved_metrics['best_val_accuracy']:.4f}  ({saved_metrics['best_val_accuracy']*100:.2f}%)")
print()
print("  Test Performance")
print(f"  |- Test accuracy    : {saved_metrics['test_accuracy']:.4f}  ({saved_metrics['test_accuracy']*100:.2f}%)")
print(f"  |- Test loss        : {saved_metrics['test_loss']:.4f}")
print(f"  |- Macro precision  : {saved_metrics['macro_precision']:.4f}")
print(f"  |- Macro recall     : {saved_metrics['macro_recall']:.4f}")
print(f"  |- Macro F1-score   : {saved_metrics['macro_f1']:.4f}")
print(f"  +- Weighted F1      : {saved_metrics['weighted_f1']:.4f}")
print()
print("  Error Analysis")
print(f"  +- Most confused: Class {CLASS_NAMES[worst_pair[0]]} -> Class {CLASS_NAMES[worst_pair[1]]}  ({int(cm_nd[worst_pair])} samples)")
print()
print(f"  Saved artifacts -> Google Drive: /efficientnet_b3_output/")
print(f"  |- best_model.keras")
print(f"  |- class_names.json")
print(f"  |- training_history.csv  +  history_all.json")
print(f"  |- test_metrics.json")
print(f"  |- confusion_matrix.png  +  confusion_matrix.json")
print(f"  |- training_curves.png")
print(f"  |- class_distribution.png")
print(f"  |- per_class_metrics.png")
print(f"  |- error_analysis.png")
print(f"  +- sample_images.png")
print(SEP)